## load libraries

In [2]:
import numpy as np
import os
import math

import torch.nn as nn
import torch
import torch.optim as optim

from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


from CNN_blocks import *
from MSEL import *
from Transformer_Encoder import *
from TS_block import *

In [3]:
device="cpu"

## load data

In [4]:
signals=np.load("../dataset/signals.npy")
labels= np.load("../dataset/labels.npy")

In [5]:
print(signals.shape)
print(labels.shape)

(64, 12, 1000)
(64, 8)


In [6]:
x=signals
y=labels

x_train, x_test, y_train, y_test = train_test_split(x, y)

mean=x_train.mean(axis=(0,2), keepdims=True)
std = x_test.std(axis=(0,2), keepdims=True)

x_train=(x_train-mean)/(std + 1e-8)
x_test =(x_test -mean)/(std + 1e-8)

x_train=torch.tensor(x_train, dtype=torch.float)
y_train=torch.tensor(y_train, dtype=torch.long)
x_test=torch.tensor(x_test, dtype=torch.float)
y_test=torch.tensor(y_test, dtype=torch.long)

In [7]:
#delete this block
tmp=x_train[-2]

In [8]:
train_dataset=TensorDataset(x_train, y_train)

In [9]:
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [10]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.CNN_blocks1=CNN_blocks1()
        self.CNN_blocks2=CNN_blocks2()

        self.lower_MSEL=lower_MSEL()
        self.lower_TS_block=lower_TS_block()  ###### parameters are remained
        
        self.upper_MSEL=upper_MSEL()
        self.upper_TS_block=upper_TS_block() ####### parameters are remained
        
        self.network=nn.Sequential(
            nn.Linear(512, 100),
            nn.ReLU(),
            
            nn.Linear(100, 80),
            nn.ReLU(),
            
            nn.Linear(80, 8)
        )

    def forward(self, x):
        
        cnn_output1=self.CNN_blocks1(x)
        cnn_output2=self.CNN_blocks2(cnn_output1)
        
        upper_MSEL_out         =  self.upper_MSEL(cnn_output2)
        upper_cls1, upper_cls2 =  self.upper_TS_block(upper_MSEL_out)
        
        lower_MSEL_out         =  self.lower_MSEL(cnn_output2)
        lower_cls1, lower_cls2 =  self.lower_TS_block(lower_MSEL_out)
        
        net_in    =  torch.cat([lower_cls2, upper_cls2], dim=1)

        
        '''
        print("x.shape:\t", x.shape, "\n")
        print("cnn_output1.shape:\t", cnn_output1.shape)
        print("cnn_output2.shape:\t", cnn_output2.shape, "\n")

        print("upper_MSEL_out.shape:\t", upper_MSEL_out.shape)
        print("lower_MSEL_out.shape:\t", lower_MSEL_out.shape, "\n")

        print("lower_cls.shape:\t", lower_cls.shape)
        print("upper_cls.shape:\t", upper_cls.shape, "\n")
        
        print("net_in.shape:\t", net_in.shape)
        '''
        
        return self.network( net_in ), [upper_cls1, lower_cls1, upper_cls2, lower_cls2]

In [11]:
model=Classifier().to(device)

In [12]:
def r2_loss(y_pred, y_true, eps=1e-8):
    ss_res = torch.sum((y_true - y_pred) ** 2, dim=1)
    mean=torch.mean(y_true, dim=1, keepdim=True)
    ss_tot = torch.sum((y_true - mean) ** 2, dim=1)
    r2 = 1 - ss_res / (ss_tot + eps)
    return   (1 - r2).mean()

In [15]:
def custom_loss(output, y_batch, cls1_u, cls1_l, cls2_u, cls2_l):
    alpha = 0.2  #defined by the paper

    bce_loss=nn.BCEWithLogitsLoss()
    first_loss=bce_loss(output, y_batch.float())
    second_loss=( r2_loss(cls2_u, cls2_l) / r2_loss(cls1_u, cls1_l) )
    total_loss = first_loss + (alpha*second_loss)
    
    return total_loss    


    

optimizer=optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [16]:
epochs=200
for epoch in range(epochs):
    model.train()
    total_loss=0
    
    for x_batch, y_batch in train_loader:
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)
        
        output, tokens = model(x_batch)
        cls1_u, cls1_l, cls2_u, cls2_l = tokens
        #output=model(x_batch)

        #loss=criterian(output, y_batch.float())
        loss=custom_loss(output, y_batch, cls1_u, cls1_l, cls2_u, cls2_l)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

        print(
            "epoch:  ", epoch, "/", epochs,"\n",
            "loss:   ", total_loss, "\n\n"
        )

/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/python3.10/site-packages/torch/cuda/__init__.py:422: UserWarning: Found GPU0 NVIDIA GeForce MX330 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.13.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/py

epoch:   0 / 200 
 loss:    0.6975377202033997 


epoch:   1 / 200 
 loss:    0.5286300778388977 


epoch:   2 / 200 
 loss:    0.26148244738578796 


epoch:   3 / 200 
 loss:    0.1636999398469925 


epoch:   4 / 200 
 loss:    0.14091472327709198 


epoch:   5 / 200 
 loss:    0.17105241119861603 


epoch:   6 / 200 
 loss:    0.14316774904727936 


epoch:   7 / 200 
 loss:    0.15313966572284698 


epoch:   8 / 200 
 loss:    0.14243240654468536 


epoch:   9 / 200 
 loss:    0.13430896401405334 


epoch:   10 / 200 
 loss:    0.13769502937793732 


epoch:   11 / 200 
 loss:    0.1335250288248062 


epoch:   12 / 200 
 loss:    0.13205905258655548 


epoch:   13 / 200 
 loss:    0.12980715930461884 


epoch:   14 / 200 
 loss:    0.13385246694087982 


epoch:   15 / 200 
 loss:    0.13070440292358398 


epoch:   16 / 200 
 loss:    0.12954846024513245 


epoch:   17 / 200 
 loss:    0.12830282747745514 


epoch:   18 / 200 
 loss:    0.12625236809253693 


epoch:   19 / 200 
 loss: 

KeyboardInterrupt: 

In [17]:
x_test=x_test.to(device)
model.eval()
with torch.no_grad():
    y_pred=model(x_test)
print(y_pred.round())

AttributeError: 'tuple' object has no attribute 'round'

In [ ]:
y_test

In [ ]:
tmp=x_train
tmp=tmp.to(device)
model.eval()
with torch.no_grad():
    y_pred=model(tmp)
print(y_pred.round())

In [ ]:
y_train